# Cheat Sheet: Multiple Linear Regression, Feature Selection, Shrinkage & Dimension Reduction

A quick-reference notebook. Each section is self-contained — copy/paste the snippet you need. Run the **Imports**
cell first; the rest can be run in any order (they don't depend on each other's variables except where noted).


## Imports (run first)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor as vif

from sklearn.linear_model import (
    LinearRegression, RidgeCV, LassoCV, ElasticNetCV, Ridge, Lasso, lasso_path
)
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.feature_selection import RFECV
from sklearn.metrics import mean_squared_error, make_scorer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


## Decision guide — which method do I reach for?

| Situation | Try this |
|---|---|
| Just fitting a first model, want interpretable coefficients | OLS (`sm.OLS` or `LinearRegression`) |
| Predictors are highly correlated with each other | Check VIF; consider Ridge or PCR |
| Want automatic variable elimination, keep interpretability | LASSO |
| Have both correlated groups AND want sparsity | Elastic Net |
| p (features) is close to or exceeds n (samples) | Ridge, LASSO, PCR, or PLS — not plain OLS |
| Want to reduce dimensionality without needing named coefficients | PCR (unsupervised components) |
| Want dimensionality reduction that's aware of the target | PLS (supervised components) |
| Need to justify feature choices to a non-technical audience | Correlation ranking + LASSO (both keep original variable names) |
| Comparing several candidate models fairly | Same train/test split, same metric, compare train vs. test error |


## 1. Fit OLS with statsmodels (get p-values, R², full summary)

In [ ]:
# X should include a constant/intercept column of 1.0 if you want an intercept term
X = sm.add_constant(X)              # or: X.insert(0, 'const', 1.0)
model = sm.OLS(y, X)
fit = model.fit()
print(fit.summary())

fit.params        # coefficients
fit.pvalues        # p-values
fit.rsquared       # R^2
fit.rsquared_adj   # adjusted R^2
fit.resid          # residuals
fit.predict(X)     # fitted/predicted values


## 2. Variance Inflation Factor (VIF) — detect & remove multicollinearity

In [ ]:
def reduce_by_vif(X_in, threshold=5.0, protect=('const',)):
    """Iteratively drop the highest-VIF column until all VIFs < threshold."""
    X_work = X_in.copy()
    log = []
    while True:
        vifs = np.array([vif(X_work.values, i) for i in range(X_work.shape[1])])
        s = pd.Series(vifs, index=X_work.columns)
        droppable = s.drop(labels=[c for c in protect if c in s.index])
        if droppable.max() < threshold:
            break
        worst = droppable.idxmax()
        log.append((worst, droppable.max()))
        X_work = X_work.drop(columns=[worst])
    return X_work, s, log

# Rule of thumb: VIF >= 5 is too high (some use 10). VIF == 1 means no correlation with other predictors.


## 3. Residual diagnostics (the 4 classical assumption checks)

In [ ]:
# Linearity: scatter each X vs y before modeling
# Normality of residuals:
plt.hist(fit.resid, bins=20)
sm.qqplot(fit.resid, line='45', fit=True)

# Homoscedasticity:
plt.scatter(fit.predict(X), fit.resid); plt.axhline(0, color='red')

# Independence / autocorrelation (only meaningful for ordered data, e.g. time series):
sm.stats.stattools.durbin_watson(fit.resid)   # ~2 => no strong autocorrelation


## 4. Correlation ranking (fast, cheap feature screen)

In [ ]:
corr_with_target = (
    df.corr()[['target_col']]
      .drop('target_col')
      .sort_values(by='target_col', key=abs, ascending=False)
)
sns.heatmap(corr_with_target, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
# NOTE: Pearson correlation is only valid for numeric/continuous features, not categorical ones.


## 5. Forward / backward selection (p-value based, from scratch)

In [ ]:
def forward_selection(X_in, y_in, threshold_in=0.05):
    remaining = [c for c in X_in.columns if c != 'const']
    selected = ['const'] if 'const' in X_in.columns else []
    while remaining:
        pvals = {c: sm.OLS(y_in, X_in[selected + [c]]).fit().pvalues[c] for c in remaining}
        best = min(pvals, key=pvals.get)
        if pvals[best] < threshold_in:
            selected.append(best); remaining.remove(best)
        else:
            break
    return selected

def backward_selection(X_in, y_in, threshold_out=0.05):
    selected = list(X_in.columns)
    while True:
        m = sm.OLS(y_in, X_in[selected]).fit()
        pvals = m.pvalues.drop(labels=['const']) if 'const' in selected else m.pvalues
        worst, worst_p = pvals.idxmax(), pvals.max()
        if worst_p > threshold_out:
            selected.remove(worst)
        else:
            break
    return selected
# Note: these greedy p-value methods tend to overfit relative to CV-based methods (see book, Ch.7).


## 6. Recursive Feature Elimination with CV (RFECV) — performance-based selection

In [ ]:
rfecv = RFECV(
    estimator=LinearRegression(),
    step=1, cv=5,
    scoring='neg_root_mean_squared_error',   # or make_scorer(your_metric, greater_is_better=False)
    min_features_to_select=1
)
rfecv.fit(X, y)
rfecv.n_features_
X.columns[rfecv.support_]          # selected feature names
-rfecv.cv_results_['mean_test_score']   # CV score per feature-count (negate if scorer was negated)


## 7. Metrics

In [ ]:
def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return 100 * np.mean(np.abs((y_true - y_pred) / y_true))

rmse(y_true, y_pred)                                  # helper defined above
mean_squared_error(y_true, y_pred)                    # MSE
np.sqrt(mean_squared_error(y_true, y_pred))           # RMSE (works on any sklearn version)


## 8. Standardize BEFORE Ridge / LASSO / Elastic Net

In [ ]:
# ALWAYS fit the scaler on TRAINING data only, then transform both train and test.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)     # .transform(), NOT .fit_transform() !


## 9. Ridge / LASSO / Elastic Net with cross-validated alpha

In [ ]:
alphas = np.logspace(-3, 3, 100)

ridge = RidgeCV(alphas=alphas, cv=5).fit(X_train_scaled, y_train)
lasso = LassoCV(alphas=alphas, cv=5, max_iter=20000).fit(X_train_scaled, y_train)
enet  = ElasticNetCV(alphas=alphas, l1_ratio=[.1,.3,.5,.7,.9,.95,.99,1],
                      cv=5, max_iter=20000).fit(X_train_scaled, y_train)

ridge.alpha_ ; lasso.alpha_ ; enet.alpha_, enet.l1_ratio_
(lasso.coef_ == 0).sum()          # how many features LASSO eliminated


### Formulas
- **Ridge (L2):** minimize `RSS + alpha * sum(beta_j^2)` — shrinks coefficients, never exactly to zero.
- **LASSO (L1):** minimize `RSS + alpha * sum(|beta_j|)` — can shrink coefficients to exactly zero (feature selection).
- **Elastic Net:** minimize `RSS + alpha * [(1-l1_ratio)/2 * sum(beta_j^2) + l1_ratio * sum(|beta_j|)]`
  — blends Ridge and LASSO; `l1_ratio=1` is pure LASSO, `l1_ratio=0` is pure Ridge.
- As `alpha -> 0`: recovers OLS. As `alpha -> infinity`: all coefficients -> 0.


## 10. Coefficient path plot (see shrinkage happen visually)

In [ ]:
alphas_path = np.logspace(-3, 1, 60)
ridge_coefs = np.array([Ridge(alpha=a).fit(X_train_scaled, y_train).coef_ for a in alphas_path])
_, lasso_coefs, _ = lasso_path(X_train_scaled, y_train, alphas=alphas_path)

plt.plot(alphas_path, ridge_coefs)     # one line per feature
plt.xscale('log'); plt.xlabel('alpha'); plt.ylabel('coefficient')


## 11. PCA / PCR — leakage-free pattern

In [ ]:
# Fit PCA on TRAINING data only:
pca = PCA()
X_pc_train = pca.fit_transform(X_train_scaled)

pca.explained_variance_ratio_          # variance explained per component
np.cumsum(pca.explained_variance_ratio_)   # cumulative variance

# Choose k via cross-validated RMSE on the training PCA scores:
cv = KFold(n_splits=10, shuffle=True, random_state=42)
scores = [-cross_val_score(LinearRegression(), X_pc_train[:, :k], y_train,
                            cv=cv, scoring='neg_root_mean_squared_error').mean()
          for k in range(1, X_pc_train.shape[1] + 1)]
best_k = int(np.argmin(scores)) + 1

# Refit and evaluate on test — TRANSFORM the test set with the TRAINING pca object:
pcr = LinearRegression().fit(X_pc_train[:, :best_k], y_train)
X_pc_test = pca.transform(X_test_scaled)[:, :best_k]     # .transform(), NOT .fit_transform() !
rmse(y_test, pcr.predict(X_pc_test))


**Common bug:** calling `pca.fit_transform(X_test)` re-fits PCA on the test set — this is data leakage and will make your test error look better than it really is. Always `.transform()` the test set using the PCA object fit on training data.

## 12. PLS Regression — supervised alternative to PCR

In [ ]:
pls_scores = [-cross_val_score(PLSRegression(n_components=k), X_train_scaled, y_train,
                                cv=cv, scoring='neg_root_mean_squared_error').mean()
              for k in range(1, min(15, X_train_scaled.shape[1]) + 1)]
best_k_pls = int(np.argmin(pls_scores)) + 1

pls = PLSRegression(n_components=best_k_pls).fit(X_train_scaled, y_train)
rmse(y_test, pls.predict(X_test_scaled))
# PLS components maximize covariance with y (supervised); PCA components only maximize variance in X (unsupervised).
# PLS often needs fewer components than PCR for comparable performance.


## 13. Categorical variables — dummy encoding

In [ ]:
dummies = pd.get_dummies(df[['cat_col_1', 'cat_col_2']], drop_first=True)
# drop_first=True picks the first category alphabetically as the reference/baseline level.
# The coefficient on each dummy = shift in mean(y) vs. the reference level, holding other vars constant.


## 14. Common pitfalls checklist

- ❌ Fitting `StandardScaler` / `PCA` on the full dataset (or the test set) before splitting → leakage.
  ✅ Fit on training data only; `.transform()` (not `.fit_transform()`) the test data.
- ❌ Using p-value-based stepwise selection as your final word on "importance" → these methods are known to
  overfit and don't correct for multiple comparisons. ✅ Prefer cross-validated performance-based selection
  (RFECV) or regularization (LASSO) for anything you'll act on.
- ❌ Comparing models evaluated on different train/test splits or different metrics → not a fair comparison.
  ✅ Fix the split and metric, then compare.
- ❌ Interpreting Ridge/PCR/PLS coefficients as "the effect of variable X" the way you would OLS coefficients →
  Ridge coefficients are biased by design; PCR/PLS coefficients apply to abstract components, not the original
  named variables (unless you back-transform them).
- ❌ Forgetting to standardize before Ridge/LASSO/Elastic Net → the penalty term is scale-sensitive; unscaled
  features with large numeric ranges will be penalized unfairly relative to small-range features.
- ❌ Using VIF or forward/backward selection on data that hasn't been checked for missing values or wrong dtypes
  first (e.g. `object` columns silently break `sm.OLS`).
